# Modelagem Preditiva - Risco de Defasagem

## Objetivo
Construir um modelo de classificacao para identificar alunos em risco academico com base nos indicadores disponiveis.

## Estrategia
- Trilha 1: todos os anos (2022-2024), sem IPP.
- Trilha 2: apenas anos com IPP (2023-2024), incluindo IPP.
- Alvo de risco: `inde_combined` no quartil inferior (Q1).
- Selecao final pelo maior `recall` em validacao, usando `roc_auc` como criterio de desempate.

## Justificativa metodologica (visao academica)
A base possui uma diferenca estrutural: o IPP nao existe em 2022. Isso cria um trade-off entre cobertura da amostra e riqueza de informacao.

- **Por que duas trilhas?**
  - Trilha 1 (sem IPP): maximiza tamanho amostral e permite comparacao longitudinal completa (2022-2024).
  - Trilha 2 (com IPP): reduz amostra, mas incorpora um indicador psicopedagogico potencialmente informativo.

- **Impacto na implementacao**
  - Mantemos dois pipelines de treino com conjuntos de features diferentes.
  - Comparamos desempenho em tabela unica, para decidir qual trilha gera melhor capacidade de deteccao de risco.
  - O app final usa o modelo vencedor, mas preservamos metadata para rastreabilidade e reproducibilidade.

## Por que `recall` e `roc_auc`?
Neste problema, falsos negativos (alunos em risco classificados como sem risco) sao mais custosos pedagogicamente.

- **Recall (sensibilidade)**
  - Definicao: `TP / (TP + FN)`.
  - Interpretacao estatistica: entre todos os casos realmente positivos, qual proporcao o modelo conseguiu recuperar.
  - Uso no projeto: metrica principal de selecao, pois prioriza deteccao de alunos em risco.

- **ROC-AUC**
  - Definicao: area sob a curva ROC (TPR vs FPR para todos os thresholds).
  - Interpretacao estatistica: probabilidade de o modelo atribuir score maior a um positivo do que a um negativo.
  - Uso no projeto: criterio de desempate por medir capacidade global de separacao, independente do limiar fixo 0.5.

## Glossario rapido das siglas
- `TP` (True Positive): caso positivo corretamente classificado como positivo.
- `FN` (False Negative): caso positivo classificado incorretamente como negativo.
- `TPR` (True Positive Rate): taxa de verdadeiros positivos; equivalente ao `recall`.
- `FPR` (False Positive Rate): taxa de falsos positivos; proporcao de negativos classificados incorretamente como positivos.

## Limitacoes metodologicas
Este estudo adota como definicao operacional de risco os alunos no quartil inferior de `inde_combined`, criterio util para priorizacao, mas dependente da distribuicao da amostra analisada. Alem disso, a ausencia estrutural de IPP em 2022 exige duas trilhas de modelagem (com e sem IPP), o que melhora a transparencia comparativa, porem introduz diferencas de cobertura temporal e tamanho amostral entre cenarios. Por fim, embora a selecao privilegie `recall` (reduzindo falsos negativos) e use `roc_auc` como desempate, a classificacao final depende de limiar de decisao (0.5 neste estudo), que pode ser recalibrado conforme o custo pedagogico de erros tipo FN e FP em implementacao real.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone

import joblib

project_root = Path.cwd().parent
normalized_path = project_root / "data" / "dados_unificados_norm.csv"
raw_path = project_root / "data" / "dados_unificados.csv"
data_path = normalized_path if normalized_path.exists() else raw_path
models_dir = project_root / "models"
outputs_dir = project_root / "outputs"
models_dir.mkdir(exist_ok=True)
outputs_dir.mkdir(exist_ok=True)

print(f"Data: {data_path}")
print(f"Models: {models_dir}")
print(f"Outputs: {outputs_dir}")

Data: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\data\dados_unificados_canonico.csv
Models: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\models
Outputs: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\outputs


In [2]:
df = pd.read_csv(data_path)

inde_cols = [c for c in ["inde_2022", "inde_2023", "inde_2024"] if c in df.columns]
for c in inde_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["inde_combined"] = df[inde_cols].mean(axis=1)
q1 = df["inde_combined"].quantile(0.25)
df["target_risco"] = (df["inde_combined"] <= q1).astype(int)

print("Shape:", df.shape)
print("Fonte:", data_path.name)
print("Anos:", sorted(df["year"].dropna().unique().tolist()))
print("Q1 inde_combined:", round(float(q1), 4))
print("Taxa de risco (%):", round(float(df["target_risco"].mean() * 100), 2))

Shape: (3030, 37)
Fonte: dados_unificados_canonico.csv
Anos: ['PEDE2022', 'PEDE2023', 'PEDE2024']
Q1 inde_combined: 6.7066
Taxa de risco (%): 20.99


In [3]:
BASE_FEATURES = [
    "phase", "age", "ian", "ida", "ieg", "iaa", "ips", "ipv",
    "math", "portuguese", "english", "deficiency", "gender",
    "school_institution", "achieved_turning_point", "indicated_for_intervention"
]


def get_existing_features(frame, include_ipp=False):
    features = BASE_FEATURES.copy()
    if include_ipp:
        features.append("ipp")
    usable = []
    for feature in features:
        if feature not in frame.columns:
            continue
        missing_rate = frame[feature].isna().mean()
        unique_values = frame[feature].nunique(dropna=True)
        if missing_rate >= 0.95 or unique_values <= 1:
            continue
        usable.append(feature)
    return usable


def evaluate_binary(y_true, y_pred, y_prob):
    metrics = {
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob))
    }
    cm = confusion_matrix(y_true, y_pred)
    metrics["tn"], metrics["fp"], metrics["fn"], metrics["tp"] = [int(x) for x in cm.ravel()]
    return metrics


def build_preprocessor(X):
    num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    pre = ColumnTransformer(transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])
    return pre


def run_track(frame, track_name, include_ipp=False):
    frame = frame.dropna(subset=["inde_combined"]).copy()
    feats = get_existing_features(frame, include_ipp=include_ipp)

    X = frame[feats].copy()
    y = frame["target_risco"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pre = build_preprocessor(X_train)

    models = {
        "logistic": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "random_forest": RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            random_state=42,
            class_weight="balanced_subsample",
            n_jobs=-1
        ),
        "gradient_boosting": GradientBoostingClassifier(random_state=42)
    }

    rows = []
    fitted = {}

    for model_name, model in models.items():
        pipe = Pipeline(steps=[
            ("pre", clone(pre)),
            ("model", model)
        ])

        pipe.fit(X_train, y_train)
        y_prob = pipe.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        metrics = evaluate_binary(y_test, y_pred, y_prob)
        row = {
            "track": track_name,
            "model": model_name,
            "n_samples": int(len(frame)),
            "n_features": int(len(feats)),
            **metrics
        }
        rows.append(row)
        fitted[model_name] = pipe

    result = pd.DataFrame(rows).sort_values(["recall", "roc_auc"], ascending=False).reset_index(drop=True)
    best_name = result.iloc[0]["model"]

    return {
        "track": track_name,
        "features": feats,
        "metrics_table": result,
        "best_model_name": best_name,
        "best_model": fitted[best_name],
        "train_size": int(len(X_train)),
        "test_size": int(len(X_test))
    }


# Auditoria de ablação para validar remoção de features temporais redundantes
ABLATION_SETS = {
    "baseline_full": [
        "year", "phase", "admission_year", "age", "age_2022", "ian", "ida", "ieg", "iaa", "ips", "ipv",
        "math", "portuguese", "english", "deficiency", "gender", "school_institution",
        "achieved_turning_point", "indicated_for_intervention", "ipp"
    ],
    "drop_age_2022": [
        "year", "phase", "admission_year", "age", "ian", "ida", "ieg", "iaa", "ips", "ipv",
        "math", "portuguese", "english", "deficiency", "gender", "school_institution",
        "achieved_turning_point", "indicated_for_intervention", "ipp"
    ],
    "drop_age_and_age_2022": [
        "year", "phase", "admission_year", "ian", "ida", "ieg", "iaa", "ips", "ipv",
        "math", "portuguese", "english", "deficiency", "gender", "school_institution",
        "achieved_turning_point", "indicated_for_intervention", "ipp"
    ],
    "core_pedagogical_only": BASE_FEATURES + ["ipp"]
}


def evaluate_feature_set(frame, feature_set):
    feats = []
    for feature in feature_set:
        if feature not in frame.columns:
            continue
        missing_rate = frame[feature].isna().mean()
        unique_values = frame[feature].nunique(dropna=True)
        if missing_rate >= 0.95 or unique_values <= 1:
            continue
        feats.append(feature)

    X = frame[feats].copy()
    y = frame["target_risco"].copy()

    pipe = Pipeline(steps=[
        ("pre", build_preprocessor(X)),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    y_prob = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=1)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    return {
        "n_features": len(feats),
        "features": feats,
        "precision": float(precision_score(y, y_pred, zero_division=0)),
        "recall": float(recall_score(y, y_pred, zero_division=0)),
        "f1": float(f1_score(y, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y, y_prob)),
        "pr_auc": float(average_precision_score(y, y_prob))
    }


ablation_base = df[df["ipp"].notna()].copy() if "ipp" in df.columns else df.copy()
ablation_rows = []
for scenario, feature_set in ABLATION_SETS.items():
    metrics = evaluate_feature_set(ablation_base, feature_set)
    ablation_rows.append({"scenario": scenario, **metrics})

ablation_report = pd.DataFrame(ablation_rows).sort_values(["recall", "roc_auc", "f1"], ascending=False).reset_index(drop=True)
ablation_report

,scenario,n_features,features,precision,recall,f1,roc_auc,pr_auc
0,drop_age_and_age_2022,15,"[year, phase, admission_year, ian, ida, ieg, i...",0.562963,0.861190,0.680851,0.924358,0.749307
1,drop_age_2022,16,"[year, phase, admission_year, age, ian, ida, i...",0.570356,0.861190,0.686230,0.924239,0.750215
2,baseline_full,17,"[year, phase, admission_year, age, age_2022, i...",0.568224,0.861190,0.684685,0.923950,0.748125
3,core_pedagogical_only,14,"[phase, age, ian, ida, ieg, iaa, ips, ipv, mat...",0.539711,0.847025,0.659316,0.917671,0.743929


## Leitura dos resultados das trilhas

Ao comparar os modelos, esta analise segue uma regra explicita de decisao:

1. Ordenar por `recall` (maior para menor), pois o objetivo principal e reduzir falsos negativos.
2. Em caso de empate, usar `roc_auc` para escolher o modelo com melhor separacao estatistica entre classes.
3. Registrar a trilha/modelo vencedor no arquivo de metadata para auditoria metodologica.

Observacao: outras metricas (`precision`, `f1`, `pr_auc`) continuam sendo reportadas para transparencia e discussao critica dos trade-offs.

In [4]:
# Trilha 1: todos os anos, com conjunto core sem IPP
track1_data = df.copy()
track1 = run_track(track1_data, track_name="trilha_core_sem_ipp", include_ipp=False)
track1["metrics_table"]

,track,model,n_samples,n_features,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,trilha_core_sem_ipp,logistic,2544,16,0.666667,0.881890,0.759322,0.937616,0.842488,326,56,15,112
1,trilha_core_sem_ipp,gradient_boosting,2544,16,0.810345,0.740157,0.773663,0.932535,0.863247,360,22,33,94
2,trilha_core_sem_ipp,random_forest,2544,16,0.801802,0.700787,0.747899,0.927969,0.853383,360,22,38,89


In [5]:
# Trilha 2: apenas linhas com IPP disponivel
if "ipp" in df.columns:
    track2_data = df[df["ipp"].notna()].copy()
    if track2_data["target_risco"].nunique() < 2:
        track2 = None
        print("Trilha 2 sem classes suficientes para classificacao.")
    else:
        track2 = run_track(track2_data, track_name="trilha_core_com_ipp", include_ipp=True)
        display(track2["metrics_table"])
else:
    track2 = None
    print("Coluna IPP nao encontrada no dataset.")

,track,model,n_samples,n_features,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,trilha_core_com_ipp,logistic,1628,14,0.588785,0.887324,0.707865,0.934769,0.812891,211,44,8,63
1,trilha_core_com_ipp,gradient_boosting,1628,14,0.681159,0.661972,0.671429,0.928031,0.796685,233,22,24,47
2,trilha_core_com_ipp,random_forest,1628,14,0.696429,0.549296,0.614173,0.921154,0.762612,238,17,32,39


In [6]:
all_tables = [track1["metrics_table"]]
candidates = [(track1["track"], track1)]

if track2 is not None:
    all_tables.append(track2["metrics_table"])
    candidates.append((track2["track"], track2))

leaderboard = pd.concat(all_tables, ignore_index=True)
leaderboard = leaderboard.sort_values(["recall", "roc_auc"], ascending=False).reset_index(drop=True)
display(leaderboard)

winner_track_name = leaderboard.iloc[0]["track"]
winner_model_name = leaderboard.iloc[0]["model"]
winner_obj = next(obj for name, obj in candidates if name == winner_track_name)
winner_model = winner_obj["best_model"]

model_path = models_dir / "model_risco.joblib"
joblib.dump(winner_model, model_path)

leaderboard_path = outputs_dir / "modelagem_leaderboard.csv"
leaderboard.to_csv(leaderboard_path, index=False, encoding="utf-8")

ablation_path = outputs_dir / "feature_ablation_report.csv"
ablation_report.to_csv(ablation_path, index=False, encoding="utf-8")

baseline_row = next(row for row in ablation_rows if row["scenario"] == "baseline_full")
recommendation = None
candidate_rows = []
for row in ablation_rows:
    recall_drop = baseline_row["recall"] - row["recall"]
    if recall_drop <= 0.01:
        candidate_rows.append((row["n_features"], -row["roc_auc"], -row["f1"], row))

if candidate_rows:
    candidate_rows.sort(key=lambda x: (x[0], x[1], x[2]))
    recommendation = candidate_rows[0][3]
else:
    recommendation = baseline_row

feature_selection_payload = {
    "dataset_rows": int(len(ablation_base)),
    "target_positive_rate": float(ablation_base["target_risco"].mean()),
    "selection_policy": "Menor conjunto de features com queda de recall <= 1pp vs baseline_full",
    "recommended_scenario": recommendation["scenario"],
    "recommended_features": recommendation["features"],
    "recommended_metrics": {
        "precision": recommendation["precision"],
        "recall": recommendation["recall"],
        "f1": recommendation["f1"],
        "roc_auc": recommendation["roc_auc"],
        "pr_auc": recommendation["pr_auc"]
    }
}

feature_selection_path = outputs_dir / "feature_selection_recommendation.json"
with open(feature_selection_path, "w", encoding="utf-8") as fp:
    json.dump(feature_selection_payload, fp, ensure_ascii=False, indent=2)

metadata = {
    "target_definition": "target_risco = 1 quando inde_combined <= Q1 (quartil inferior)",
    "q1_inde_combined": float(q1),
    "winner_track": winner_track_name,
    "winner_model": winner_model_name,
    "winner_metrics": leaderboard.iloc[0].to_dict(),
    "track1": {
        "features": track1["features"],
        "train_size": track1["train_size"],
        "test_size": track1["test_size"]
    },
    "track2": None if track2 is None else {
        "features": track2["features"],
        "train_size": track2["train_size"],
        "test_size": track2["test_size"]
    }
}

metadata_path = outputs_dir / "model_risco_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as fp:
    json.dump(metadata, fp, ensure_ascii=False, indent=2)

print(f"Modelo salvo em: {model_path}")
print(f"Leaderboard salvo em: {leaderboard_path}")
print(f"Ablation report salvo em: {ablation_path}")
print(f"Feature selection salvo em: {feature_selection_path}")
print(f"Metadata salvo em: {metadata_path}")
print(f"Vencedor: {winner_track_name} / {winner_model_name}")

,track,model,n_samples,n_features,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,trilha_core_com_ipp,logistic,1628,14,0.588785,0.887324,0.707865,0.934769,0.812891,211,44,8,63
1,trilha_core_sem_ipp,logistic,2544,16,0.666667,0.881890,0.759322,0.937616,0.842488,326,56,15,112
2,trilha_core_sem_ipp,gradient_boosting,2544,16,0.810345,0.740157,0.773663,0.932535,0.863247,360,22,33,94
3,trilha_core_sem_ipp,random_forest,2544,16,0.801802,0.700787,0.747899,0.927969,0.853383,360,22,38,89
4,trilha_core_com_ipp,gradient_boosting,1628,14,0.681159,0.661972,0.671429,0.928031,0.796685,233,22,24,47
5,trilha_core_com_ipp,random_forest,1628,14,0.696429,0.549296,0.614173,0.921154,0.762612,238,17,32,39


Modelo salvo em: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\models\model_risco.joblib
Leaderboard salvo em: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\outputs\modelagem_leaderboard.csv
Ablation report salvo em: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\outputs\feature_ablation_report.csv
Feature selection salvo em: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\outputs\feature_selection_recommendation.json
Metadata salvo em: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\outputs\model_risco_metadata.json
Vencedor: trilha_core_com_ipp / logistic


## Proximos passos
1. Validar o app Streamlit com o modelo vencedor salvo em `models/model_risco.joblib`.
2. Consolidar o storytelling final usando `outputs/modelagem_leaderboard.csv` e `outputs/model_risco_metadata.json`.
3. Se houver tempo, testar validacao temporal como extensao metodologica.